# Qwen2.5-1.5B-Instruct, untrained (zero-shot), on all three kinds of data
Same script, same prompt (all 187 names with definitions) and same sets as the earlier Qwen2.5-7B-Instruct zero-shot run, so the two models can be compared directly.

Sets: our generated eval (fallacy examples and the "None" set), CoCoLoFa (calibration and test slices, plus the Appeal to Tradition slice) and MAFALDA.

**Before you start**
- Upload `fallacy_code.zip` and `external_eval.zip` (CoCoLoFa and MAFALDA test files, private, local use only) to the Files sidebar and wait for both uploads to finish.
- The Colab secret `HF_TOKEN` (Hugging Face read token) must exist with *Notebook access* on. It is used to download our generated data.
- Any GPU works. The model is small and runs at full precision. Expect about 10-15 minutes.

In [ ]:
!nvidia-smi
!pip -q uninstall -y torchao
!pip -q install peft transformers accelerate tqdm huggingface_hub sentencepiece
!pip list 2>/dev/null | grep -iE "^(transformers|torch|accelerate) "

In [ ]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN').strip()  # the secret can only be read from the notebook process
!ls -la /content/*.zip
!rm -rf /content/work && mkdir -p /content/work
!for z in /content/fallacy_code.zip /content/external_eval.zip; do unzip -tq $z || echo "BROKEN (incomplete upload): $z"; unzip -oq $z -d /content/work; done
%cd /content/work
!python fetch_data.py --rev v5
!wc -l data/generated/eval.jsonl data/generated/eval_none.jsonl external/*.jsonl

## Run: Qwen2.5-1.5B-Instruct, zero-shot, all sets
Timestamped messages appear as it works. Each set ends with a line that starts with the set name, then `n=`, `top-1`, `None recall` and `fallacy top-1`.

In [ ]:
!python -u zeroshot_open.py --model Qwen/Qwen2.5-1.5B-Instruct 2>&1 | tee zs_qwen1.5b.log

## Download the results

In [ ]:
!zip -q zeroshot_qwen1.5b_results.zip data/zeroshot_*.json zs_qwen1.5b.log
from google.colab import files
files.download('/content/work/zeroshot_qwen1.5b_results.zip')